In [ ]:
import pandas as pd

import os
import s3fs


fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio.lab.sspcloud.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])

"""
with fs.open("s3://lab/art_net_dec.parquet") as f : 
    dfa = pd.read_parquet(f)

with fs.open("s3://lab/conf_net_dec.parquet") as f : 
    dfc = pd.read_parquet(f)"""

#with fs.open("s3://lab/mem/full_name_list.csv") as f : 
   # nl = pd.read_csv(f, index_col=[0])
"""
with fs.open("s3://lab/mem/n_to_g.csv") as f : 
    ntg = pd.read_csv(f, index_col=[0])"""

with fs.open("s3://lab/kgnd.csv") as f:
    kgnd = pd.read_csv(f)
with fs.open("s3://lab/ignd.csv") as f:
    ignd = pd.read_csv(f)
with fs.open("s3://lab/wgnd.csv") as f:
    wgnd = pd.read_csv(f)
with fs.open("s3://lab/jgnd.csv") as f:
    jgnd = pd.read_csv(f)
with fs.open("s3://lab/cgnd.csv") as f:
    cgnd = pd.read_csv(f)
with fs.open("s3://lab/usgnd.csv") as f: 
    usgnd = pd.read_csv(f)

GND preprocessing

In [2]:
gnd=pd.concat([wgnd,cgnd,kgnd,ignd,jgnd,usgnd],axis=0,ignore_index=True)
gnd=gnd.drop(columns=["Count"])
gnd["name"] = gnd["Name"].str.lower()
gnd = gnd.drop("Name",axis=1)
#gnd = gnd.drop_duplicates()
gnd

,Gender,name
0,F,baby
1,F,aisyah
2,F,anela
3,F,anela
4,F,fiyinfoluwa
...,...,...
27351475,M,zylyn
27351476,M,zymiere
27351477,M,zypher
27351478,M,zyre


In [ ]:
import re, string

punct = re.escape(string.punctuation)
gnd["name"] = gnd["name"].str.replace(f'[{punct}]', '', regex=True)

In [ ]:
# mode or M
def mode_with_tiebreak(x):
    counts = x.value_counts()
    if counts.empty:
        return None
    max_count = counts.max()
    top = counts[counts == max_count].index
    return "M" if "M" in top else top[0]

gnd = gnd.groupby("name")["Gender"].agg(mode_with_tiebreak).reset_index()

In [ ]:
#duplicates
pd.concat(g for _, g in gnd.groupby("name") if len(g) > 1)

,Gender,name
24295545,M,(shaurya)
24296394,M,(shaurya)
24297242,M,(shaurya)
24298089,M,(shaurya)
24298935,M,(shaurya)
...,...,...
23733214,M,ハ
23919591,M,ハ
22735183,M,バ
23123722,M,バ


NL prepro

In [8]:
dfa["authors"] = dfa["authors"].apply(lambda lst: [d["name"] for d in lst])
dfc["authors"] = dfc["authors"].apply(lambda lst: [d["name"] for d in lst])

In [16]:
nla = dfa["authors"].explode().str.split(' ').str[0]
nlc = dfc["authors"].explode().str.split(' ').str[0]

In [19]:
nl = pd.concat([nla,nlc]).drop_duplicates()

In [17]:
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z]\.$", na=False)]
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z]\.-[A-Za-z]\.$", na=False)]
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z]-[A-Za-z]\.$", na=False)]
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z][A-Za-z]\.$", na=False)]
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z][A-Za-z][A-Za-z]\.$", na=False)]

In [ ]:
# for jean-marie
nl = nl["authors"].str.replace("-", " ").str.split(" ").str[0].str.lower()

In [51]:
nl.to_csv("s3://lab/mem/full_name_list.csv")

NTG prepro

In [ ]:
ntg = pd.DataFrame(nl).merge(gnd,how="left",left_on="authors",right_on="Name")

In [43]:
ntg[~ntg["Gender"].isna()]

,authors,Name,Gender
0,kai,kai,M
1,richard,richard,M
2,stefan,stefan,M
3,herbert,herbert,M
4,sophie,sophie,F
...,...,...,...
313617,chien,chien,M
313625,chiu,chiu,M
313628,tasi,tasi,M
313629,xiangxu,xiangxu,M


In [37]:
gnd = gnd.drop_duplicates(subset="Name")

In [53]:
!pip install gender-guesser

In [71]:
import gender_guesser.detector as gg

d = gg.Detector()
ng_dict = pd.DataFrame()
ng_dict["name"] = nl
ng_dict["gender"] = nl.apply(lambda x: d.get_gender(x))
ng_dict

,name,gender
0,Kai,male
1,Richard,male
1,Stefan,male
2,Herbert,male
3,Sophie,female
...,...,...
3749381,Siwaphon,unknown
3749388,Tasi-Ying,unknown
3749413,XiangXu,unknown
3749415,Pradhyumansinh,unknown


In [72]:
ng_dict[ng_dict["gender"]=="unknown"]

,name,gender
4,NaN,unknown
27,Hans-Jürgen,unknown
42,Nicolaos,unknown
87,Brikenrikena,unknown
96,Catalin,unknown
...,...,...
3749381,Siwaphon,unknown
3749388,Tasi-Ying,unknown
3749413,XiangXu,unknown
3749415,Pradhyumansinh,unknown


In [44]:
ntg.to_csv("s3://lab/mem/n_to_g.csv")

In [4]:
#dfa["authors"] = dfa["authors"].apply(lambda lst: [d["name"] for d in lst])
attr = dfa.explode("authors")
#attr["name"] = attr["authors"].str.split(" ").str[0].str.lower()
#attr = attr.merge(ntg[["authors","Gender"]],how="left",left_on="name",right_on="authors")
attr

,title,journal,year,authors,dblp_uri,doi
0,Auswirkung der Digitalisierung auf die Systeml...,Elektrotech. Informationstechnik,2019,Kai Schlabitz,journals/ei/Schlabitz19,https://doi.org/10.1007/s00502-019-0687-y
1,EMF-Personenschutz: Neue Aspekte in der numeri...,Elektrotech. Informationstechnik,2020,Richard Überbacher,journals/ei/UberbacherC20,https://doi.org/10.1007/s00502-020-00791-z
1,EMF-Personenschutz: Neue Aspekte in der numeri...,Elektrotech. Informationstechnik,2020,Stefan Cecil,journals/ei/UberbacherC20,https://doi.org/10.1007/s00502-020-00791-z
2,Zur Genesis der Forschungsstelle für Integrier...,Elektrotech. Informationstechnik,2022,Herbert Mang,journals/ei/Mang22,https://doi.org/10.1007/s00502-022-01049-6
3,100 % erneuerbare Energie für Österreichs Indu...,Elektrotech. Informationstechnik,2021,Sophie Knöttner,journals/ei/KnottnerGDD21,https://doi.org/10.1007/s00502-021-00953-7
...,...,...,...,...,...,...
3992220,Analysis of projected hydrological behavior of...,Hydrology and Earth System Sciences,2012,Rita Ley,persons/CasperGGGHLR12,https://www.wikidata.org/entity/Q114958217
3992220,Analysis of projected hydrological behavior of...,Hydrology and Earth System Sciences,2012,Andreas Rock,persons/CasperGGGHLR12,https://www.wikidata.org/entity/Q114958217
3992221,Common Subexpression Identification in General...,"Technical Rep. UKSC 0060, IBM United Kingdom S...",1974,Patrick A. V. Hall,persons/Hall74,None
3992222,Interactive Support for Non-Programmers: The R...,"Research Report / RJ / IBM / San Jose, California",1974,E. F. Codd,persons/CoddD74,None


In [ ]:
attr = attr.groupby("authors", as_index=False).agg({col: list for col in attr.columns if col != "authors"})

In [ ]:
attr

In [ ]:
import string

attr["authors"] = attr["authors"].str.strip(string.punctuation)
attr["name"] = attr["authors"].str.split(" ").str[0].str.lower()
#attr = attr.merge(ntg[["Name","Gender"]],how="left",left_on="name",right_on="Name")
#attr = attr.drop(df.columns["Name"], axis = 1)
attr

In [ ]:
attr.to_parquet("s3://lab/mem/art_attr.parquet")